# C2 synthetic transcripts (P2-e, ADR 0021)

Generates the 400 C2 transcripts with **Qwen/Qwen3.5-9B** (Apache-2.0) at commit `c202236`, 4-bit
NF4 on a free **T4** runtime. Only synthetic prompts (`resources/c2/prompts.jsonl`) and the
generated transcripts enter this notebook; no project data, no keys.

- The generator and the prompts are fetched from the repository at a **pinned commit**.
- The model's 14 files are checked against the hashes Hugging Face lists for that commit;
  any mismatch stops the run.
- Output goes to **Google Drive** (`MyDrive/presenceai_c2/`), one line per transcript, flushed
  after every batch of 5. A new run skips finished prompts, so a disconnect loses at most the
  batch in progress.
- Long jobs run in the background (`subprocess.Popen`) and write a log; the status cell reads it.

Run the cells in order. Cells 1-2 need you (runtime type, Drive sign-in).

In [ ]:
# 1. Runtime check: needs a T4 GPU (Runtime > Change runtime type > T4 GPU).
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "NO GPU - switch the runtime to T4")

In [ ]:
# 2. Mount Google Drive (asks for your sign-in). Output survives a runtime disconnect.
from google.colab import drive
drive.mount("/content/drive")
import pathlib
OUT = pathlib.Path("/content/drive/MyDrive/presenceai_c2"); OUT.mkdir(parents=True, exist_ok=True)
print(OUT, sorted(p.name for p in OUT.iterdir()))

In [ ]:
# 3. Fetch the generator and the synthetic prompts at a pinned commit; install libraries (background).
import subprocess, urllib.request, pathlib, hashlib
COMMIT = "b12f6f1de3c48ae9fffaf15451884c64cb351d3b"
BASE = f"https://raw.githubusercontent.com/hemobemoo/PresenceAI/{COMMIT}"
work = pathlib.Path("/content/c2"); work.mkdir(exist_ok=True)
for src, dst in [("experiments/c2_synth/generate_c2.py", "generate_c2.py"),
                 ("resources/c2/prompts.jsonl", "prompts.jsonl")]:
    urllib.request.urlretrieve(f"{BASE}/{src}", work / dst)
    print(dst, hashlib.sha256((work / dst).read_bytes()).hexdigest()[:16])
jobs = {}
jobs["pip"] = subprocess.Popen(["pip", "install", "-q", "-U", "transformers", "accelerate",
                                "bitsandbytes", "huggingface_hub"],
                               stdout=open(work / "pip.log", "w"), stderr=subprocess.STDOUT)
print("pip started")

In [ ]:
# 4. Download Qwen3.5-9B at commit c202236 and verify every file's hash (background; ~18 GB).
jobs["fetch"] = subprocess.Popen(["python", "generate_c2.py", "fetch", "--model-dir", "/content/qwen35-9b"],
                                 cwd=work, stdout=open(work / "fetch.log", "w"), stderr=subprocess.STDOUT)
print("fetch started")

In [ ]:
# 5. Status: background jobs and the tail of each log. Re-run as often as you like.
import time
def status(wait_s=20):
    t0 = time.time()
    while any(j.poll() is None for j in jobs.values()) and time.time() - t0 < wait_s:
        time.sleep(1)
    for name, j in jobs.items():
        log = work / f"{name}.log"
        tail = log.read_text(errors="replace").splitlines()[-4:] if log.exists() else []
        print(f"== {name}: {'running' if j.poll() is None else f'exit {j.returncode}'}")
        print("\n".join(tail))
status()

In [ ]:
# 6. Pilot 5 (owner, 2026-11-22): tightened no-conclusion prompt, 3 new topics (04, 05, 06).
ids = ",".join(f"c2_no_conclusion_{t:02d}" for t in (4, 5, 6))
jobs["pilot5"] = subprocess.Popen(["python", "generate_c2.py", "run", "--model-dir", "/content/qwen35-9b",
                                   "--prompts", "prompts.jsonl", "--out", str(OUT / "pilot5"),
                                   "--ids", ids, "--batch", "3"],
                                  cwd=work, stdout=open(work / "pilot5.log", "w"), stderr=subprocess.STDOUT)
print("pilot 5 started")

In [ ]:
# 7. Pilot 5: the last two sentences of each transcript, and the attempts.
import json, re
for f in ("attempts", "generated", "rejected"):
    p = OUT / "pilot5" / f"{f}.jsonl"
    rows = [json.loads(x) for x in p.read_text().splitlines() if x.strip()] if p.exists() else []
    print(f"{f}: {len(rows)}")
    if f == "generated":
        for r in rows:
            sents = re.split(r"(?<=[.!?])\s+", r["text"].strip())
            print(f"\n{r['id']} (attempt {r['attempt']}, {r['words']} words):\n  " + "\n  ".join(sents[-2:]))

In [ ]:
# 8. Full run: all 400 prompts, batches of 5, at most 3 attempts each. Resumes after a disconnect:
#    re-run cells 1-5 (model files are re-fetched and re-verified), then this cell.
jobs["full"] = subprocess.Popen(["python", "generate_c2.py", "run", "--model-dir", "/content/qwen35-9b",
                                 "--prompts", "prompts.jsonl", "--out", str(OUT / "full"), "--batch", "5"],
                                cwd=work, stdout=open(work / "full.log", "a"), stderr=subprocess.STDOUT)
print("full run started")

In [ ]:
# 9. Full-run progress from the files on Drive (works even if the log is gone).
from collections import Counter
def count(name):
    p = OUT / "full" / f"{name}.jsonl"
    return [json.loads(x) for x in p.read_text().splitlines() if x.strip()] if p.exists() else []
gen, rej, att = count("generated"), count("rejected"), count("attempts")
print(f"accepted {len(gen)}, rejected {len(rej)}, finished {len(gen) + len(rej)}/400, attempts {len(att)}")
print("accepted by group", dict(Counter(r["group"] for r in gen)))
print("failed attempts by group", dict(Counter(r["group"] for r in att if r["problems"])))